# 5. Results and Economic Interpretation

We interpret the final regression estimates, their uncertainty and what they tell us about differences in parking occupancy across Los Angeles.

<a id="current-5a"></a>

## 5(a) Final Regression Results

The final model selected in [Section 3](03_variable_selection_models.ipynb#current-3f) uses 4,016 parking spaces across 45 Census tracts. The dependent variable is time-weighted August 2025 occupancy, measured as a fraction from zero to one. We keep the same income-complete sample used to compare the models, even though income is not a final predictor.

The table reports OLS coefficients with HC1 heteroskedasticity-robust standard errors, p-values and 95% confidence intervals. The variance tests in [Section 4](04_diagnostics_robustness.ipynb#current-4b) reject constant error variance. HC1 adjusts uncertainty without changing the estimated coefficients.

In [1]:
import pandas as pd
import statsmodels.api as sm

parking = pd.read_csv(
    "../data/processed/parking_cross_section.csv",
    dtype={"SpaceID": str, "GEOID": str}
)
parking = parking[parking["RateType"].isin(["FLAT", "TOD"])].copy()
parking["RateType_TOD"] = (parking["RateType"] == "TOD").astype(int)

predictors = [
    "price_midpoint", "population_density", "nearby_restaurant_retail_count",
    "transit_share", "walk_share", "nearby_business_count", "RateType_TOD"
]
data = parking[["occupancy_rate"] + predictors + ["median_income"]].dropna()
X = sm.add_constant(data[predictors])
y = data["occupancy_rate"]

model = sm.OLS(y, X).fit()
robust = sm.OLS(y, X).fit(cov_type="HC1")


In [2]:
regression_table = pd.DataFrame({
    "Coefficient": model.params,
    "HC1 standard error": robust.bse,
    "p-value": robust.pvalues,
    "95% lower": robust.conf_int()[0],
    "95% upper": robust.conf_int()[1]
})
regression_table.index = [
    "Intercept", "Price (dollars/hour)", "Population density (people/sq mile)",
    "Food-service/retail registrations", "Transit share", "Walk share",
    "All business registrations", "TOD (FLAT reference)"
]

print("Observations:", int(model.nobs))
regression_table.style.format({
    "Coefficient": "{:.6g}", "HC1 standard error": "{:.6g}",
    "p-value": "{:.2e}", "95% lower": "{:.6g}", "95% upper": "{:.6g}"
})


Observations: 4016


,Coefficient,HC1 standard error,p-value,95% lower,95% upper
Intercept,0.319517,0.0105497,1.71e-201,0.29884,0.340194
Price (dollars/hour),0.0515432,0.00224949,3.43e-116,0.0471343,0.0559521
Population density (people/sq mile),3.96115e-06,2.27304e-07,5.18e-68,3.51564e-06,4.40665e-06
Food-service/retail registrations,-0.000534366,3.8711e-05,2.41e-43,-0.000610238,-0.000458494
Transit share,0.100461,0.0459653,2.88e-02,0.0103709,0.190552
Walk share,0.282333,0.0267372,4.59e-26,0.229929,0.334737
All business registrations,6.66312e-05,8.88582e-06,6.45e-14,4.92153e-05,8.40471e-05
TOD (FLAT reference),0.0435671,0.00809947,7.49e-08,0.0276925,0.0594418


<a id="current-5b"></a>

## 5(b) Economic Interpretation of the Coefficients

**Price.** Holding the other included variables constant, a one-dollar higher hourly price proxy is associated with **5.15 percentage points higher occupancy**. This is a difference in the occupancy rate, not a 5.15% relative change. Prices can be higher where demand is stronger, and the inventory prices postdate the occupancy month, so this is not evidence that raising prices increases parking demand.

**Population density.** An additional **1,000 people per square mile** is associated with **0.40 percentage points higher occupancy**, holding the included variables constant. This is consistent with denser areas having stronger demand, although the estimate does not isolate a causal density effect.

**Business activity.** An additional **100 food-service/retail registrations** is associated with **5.34 percentage points lower occupancy when total business registrations are held fixed**. This comparison describes a different mix of businesses, not an increase in the overall number of businesses. In contrast, **100 additional total registrations**, holding food-service/retail registrations fixed, is associated with **0.67 percentage points higher occupancy**. Because registrations can share addresses, these changes should not be read as 100 distinct new storefronts.

**Commuting shares.** A **10-percentage-point higher transit share** is associated with **1.00 percentage point higher occupancy**; the same increase in walking share is associated with **2.82 percentage points higher occupancy**. These shares describe tract residents, not necessarily the people using the parking spaces. They may capture differences in centrality and neighborhood demand as well as travel behavior.

**TOD pricing.** TOD spaces have **4.36 percentage points higher predicted occupancy** than FLAT spaces with the same included covariates. This is a conditional comparison between tariff structures, not the effect of switching an individual meter to TOD pricing.

**Intercept.** The intercept is **0.3195**, or **31.95% occupancy**, for a FLAT space when every numeric predictor is zero. That combination lies outside the observed data, so the intercept has little standalone economic meaning.

<a id="current-5c"></a>

## 5(c) Statistical and Economic Significance

All seven slopes are statistically significant at the 5% level using HC1 standard errors, but their economic importance depends on the size and units of the comparison. The price difference of 5.15 occupancy percentage points per dollar is substantial relative to mean sample occupancy of about 53.76%. The density comparison is smaller: 0.40 percentage points for another 1,000 people per square mile.

Transit share has the weakest statistical evidence, with a p-value of about 0.029, and its estimated one-percentage-point occupancy difference is less precise than several of the other comparisons. Small p-values alone do not establish large policy benefits or causal effects. The estimates also follow variable selection on the same sample, so their reported significance is conditional on the selected specification.

<a id="current-5d"></a>

## 5(d) Confidence Intervals

In [3]:
changes = pd.Series({
    "price_midpoint": 1,
    "population_density": 1000,
    "nearby_restaurant_retail_count": 100,
    "transit_share": 0.10,
    "walk_share": 0.10,
    "nearby_business_count": 100,
    "RateType_TOD": 1
})

economic_intervals = pd.DataFrame({
    "Comparison": [
        "One-dollar higher price", "1,000 more people/sq mile",
        "100 more food/retail registrations, total fixed", "10-pp higher transit share",
        "10-pp higher walk share", "100 more total registrations, food/retail fixed",
        "TOD versus FLAT"
    ],
    "Occupancy difference (pp)": robust.params[predictors] * changes * 100,
    "95% lower (pp)": robust.conf_int().loc[predictors, 0] * changes * 100,
    "95% upper (pp)": robust.conf_int().loc[predictors, 1] * changes * 100
})
economic_intervals.style.format({
    "Occupancy difference (pp)": "{:.3f}",
    "95% lower (pp)": "{:.3f}", "95% upper (pp)": "{:.3f}"
})


,Comparison,Occupancy difference (pp),95% lower (pp),95% upper (pp)
price_midpoint,One-dollar higher price,5.154,4.713,5.595
population_density,"1,000 more people/sq mile",0.396,0.352,0.441
nearby_restaurant_retail_count,"100 more food/retail registrations, total fixed",-5.344,-6.102,-4.585
transit_share,10-pp higher transit share,1.005,0.104,1.906
walk_share,10-pp higher walk share,2.823,2.299,3.347
nearby_business_count,"100 more total registrations, food/retail fixed",0.666,0.492,0.840
RateType_TOD,TOD versus FLAT,4.357,2.769,5.944


The HC1 95% interval for the price association is **4.71 to 5.60 occupancy percentage points per dollar**. For a 1,000-person increase in population density, it is **0.35 to 0.44 percentage points**. These intervals exclude zero under the fitted inference assumptions.

The transit-share interval is wider relative to its estimate: a 10-percentage-point higher share is associated with **0.10 to 1.91 percentage points higher occupancy**. The evidence supports a positive conditional association, but its economic size remains uncertain.

These are nominal 95% confidence intervals, not 95% probabilities that fixed coefficients fall within the reported bounds. HC1 addresses heteroskedasticity but not correlated errors across nearby spaces, model-selection uncertainty or the functional-form concerns found in Section 4. Those limitations prevent treating the intervals as proof of causal effects.

<a id="current-5e"></a>

## 5(e) Model Fit and Economic Context

In [4]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

folds = KFold(n_splits=5, shuffle=True, random_state=10)
cv_rmse = -cross_val_score(
    LinearRegression(), data[predictors], y,
    cv=folds, scoring="neg_root_mean_squared_error"
)

pd.Series({
    "Observations": int(model.nobs),
    "R-squared": model.rsquared,
    "Adjusted R-squared": model.rsquared_adj,
    "AIC": model.aic,
    "BIC": model.bic,
    "Mean five-fold RMSE": cv_rmse.mean()
}).round(5)


Observations           4016.00000
R-squared                 0.29414
Adjusted R-squared        0.29291
AIC                   -3185.76999
BIC                   -3135.38566
Mean five-fold RMSE       0.16280
dtype: float64

The model explains **29.41%** of the sample variation in monthly occupancy, with an adjusted R² of **0.2929**. About 70.59% remains unexplained, which is substantial variation across spaces despite the included neighborhood and pricing measures. Omitted parking supply, enforcement, local destinations and reporting differences may contribute to that variation.

Mean five-fold RMSE is **0.16280**, or about **16.28 occupancy percentage points**. The model improves on the simpler specifications compared in Section 3, but that error is large enough to limit precise predictions for individual spaces. Random folds can include neighboring spaces and the same tracts in both training and test sets. These results measure held-out-space prediction in the observed setting, not performance in new neighborhoods or future months.

### Comparison with the Literature

Shoup (2006) explains how the price difference between curb and off-street parking can encourage cruising when curb spaces are scarce. Our positive price association does not isolate that demand response: high-demand locations can have both higher prices and higher occupancy. We also measure monthly occupancy rather than cruising time, so the estimate does not directly test his search-cost mechanism.

Pierce and Shoup (2013) report an average price elasticity of about −0.4 from SFpark price adjustments. Our coefficient instead measures a level difference in occupancy per dollar across Los Angeles spaces. The different units, study designs and source dates mean that a positive cross-sectional association should not be read as an opposing causal elasticity. The comparison supports studying historically aligned tariff changes before drawing pricing-policy conclusions.

### Conclusion

Higher stated prices, greater population density, higher resident transit and walking shares, and TOD pricing are associated with higher occupancy in the final model. The business-count estimates describe conditional differences in commercial composition. The model captures some meaningful spatial variation, but its prediction error, specification concerns and observational design limit its use for causal pricing recommendations.

### References

Shoup, D. C. (2006). Cruising for parking. *Transport Policy, 13*(6), 479–486. [doi:10.1016/j.tranpol.2006.05.005](https://doi.org/10.1016/j.tranpol.2006.05.005).

Pierce, G., & Shoup, D. (2013). Getting the prices right: An evaluation of pricing parking by demand in San Francisco. *Journal of the American Planning Association, 79*(1), 67–81. [doi:10.1080/01944363.2013.787307](https://doi.org/10.1080/01944363.2013.787307).